# 📊 Phần 2: Ứng Dụng Dữ Liệu Thực Tế - Dự Đoán Mức Lương NBA

Notebook này thực hiện toàn bộ quy trình từ Khảo sát dữ liệu (EDA), Tiền xử lý dữ liệu hướng đối tượng (OOP Data Pipeline), Huấn luyện & so sánh các mô hình hồi quy (OLS, OLS chọn biến, Ridge, Lasso, Bayesian Linear Regression) và trực quan hóa kết quả.

## 1. Khảo sát dữ liệu (Exploratory Data Analysis - EDA)

Chúng ta sẽ đọc dữ liệu từ file csv và khảo sát sơ bộ:
- Kích thước dữ liệu
- Tỷ lệ giá trị khuyết (Missing Values)
- Phân phối mức lương cầu thủ (Salary) và lý do sử dụng phép biến đổi Logarit $y' = \ln(\text{Salary})$

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd())))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

data_path = 'part2/data/nba_salary_raw.csv'
if not os.path.exists(data_path):
    data_path = 'part2/data/nba_salaries.csv'
df = pd.read_csv(data_path)

print("📊 Kích thước dữ liệu:", df.shape)
print("\n--- Tỷ lệ giá trị khuyết tự nhiên trong từng đặc trưng ---")
missing = df.isnull().mean()
print(missing[missing > 0])

# Trực quan phân phối mức lương
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Salary'] / 1e6, kde=True, ax=axes[0], color='tomato')
axes[0].set_xlabel('Salary (Millions USD)')
axes[0].set_title('Phân phối Lương (Bị lệch phải nặng)')

sns.histplot(np.log(df['Salary']), kde=True, ax=axes[1], color='forestgreen')
axes[1].set_xlabel('Log Salary')
axes[1].set_title('Phân phối Log Lương (Phân phối chuẩn hơn)')
plt.tight_layout()
plt.show()

## 2. Tiền xử lý dữ liệu (OOP Data Pipeline)

Chúng ta chạy class `NBADataPipeline` thiết kế từ `part2/data_pipeline.py`. Quy trình tiền xử lý tuân thủ:
1. Chia Train/Test riêng biệt trước để chống rò rỉ dữ liệu (Data Leakage).
2. Chuẩn hóa đặc trưng số (Standardization) dựa trên thông số từ tập Train.
3. Điền khuyết bằng thuật toán K-NN (KNN Imputer) trên thang đo chuẩn hóa để tính khoảng cách chính xác.
4. Encode biến phân loại (Dummy Variables) và căn chỉnh tập Test trùng khớp tập Train.

In [ ]:
from part2.data_pipeline import NBADataPipeline

# Chia tập dữ liệu
train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)

# Thực thi quy trình
pipeline = NBADataPipeline()
X_train, y_train, X_test, y_test = pipeline.process_pipeline(train_df, test_df)

print(f"Kích thước dữ liệu huấn luyện X_Train: {X_train.shape} | Y_Train: {y_train.shape}")
print(f"Kích thước dữ liệu kiểm thử    X_Test:  {X_test.shape}  | Y_Test:  {y_test.shape}")
print(f"Còn NaN trong tập huấn luyện? ->", X_train.isnull().sum().sum() > 0)

## 3. Huấn luyện và So sánh các mô hình

Chúng ta sẽ thực thi toàn bộ script `part2/model_comparison.py` để:
- Huấn luyện 5 mô hình (OLS, OLS chọn biến, Ridge, Lasso, Bayesian)
- Đánh giá mô hình trên tập Test riêng biệt thông qua các chỉ số MAE, RMSE, R2 trên thang đo gốc (USD)
- Vẽ đồ thị phân tích chẩn đoán phần dư
- Vẽ đồ thị khoảng tin cậy Bayesian

In [ ]:
from part2.model_comparison import main
main()

## 4. Hiển thị trực tiếp các kết quả trực quan hóa

Dưới đây là các hình ảnh biểu đồ phân tích và kết quả dự đoán được sinh ra từ quy trình so sánh mô hình:

In [ ]:
from IPython.display import Image, display
import os

# 1. Đồ thị chẩn đoán phần dư của mô hình tốt nhất
if os.path.exists('residuals_diagnostic.png'):
    print("1. Phân Tích Chẩn Đoán Phần Dư (Mô hình tốt nhất)")
    display(Image('residuals_diagnostic.png'))

# 2. Ridge Trace
if os.path.exists('ridge_trace.png'):
    print("\n2. Ridge Trace Analysis")
    display(Image('ridge_trace.png'))

# 3. Bayesian Prediction Intervals
if os.path.exists('bayesian_prediction_intervals.png'):
    print("\n3. Khoảng Dự Đoán Tin Cậy Bayesian 95%")
    display(Image('bayesian_prediction_intervals.png'))